# Colab smoke test 1: collapsed Answers and Run all

Open this notebook **from GitHub** (the colab.research.google.com/github/... link), not from a downloaded copy.
Follow the numbered checks in the professor's checklist (R1 to R12). Do not expand anything before check R1.

Each test has a question (code shown as text), then an **Answer** heading whose section holds the expected output
and a code cell that runs the question's code. After each test there is a visible **MARKER** cell: after
**Runtime > Run all**, the last MARKER that shows output tells how far Run all got.

## Setup

In [ ]:
import sys, IPython, ipykernel
print("Python", sys.version.split()[0], "| IPython", IPython.__version__, "| ipykernel", ipykernel.__version__)

from os.path import exists
from urllib.request import urlretrieve
if not exists('jupyturtle.py'):
    urlretrieve('https://github.com/ramalho/jupyturtle/releases/download/2024-03/jupyturtle.py', 'jupyturtle.py')
from jupyturtle import make_turtle, forward, left


def run_code(code):
    """A: run code (a string) as if it were a cell of its own; an error is shown as a normal
    (red) error output, but the cell itself succeeds, so "Run all" goes on."""
    try:
        ip = get_ipython()
    except NameError:              # plain Python: just run it
        exec(code, globals())
        return
    result = ip.run_cell(code, store_history=False)
    if isinstance(result.error_in_exec, KeyboardInterrupt):
        raise KeyboardInterrupt    # the Stop button still stops "Run all"


def run_code_stderr(code):
    """B (fallback 1): the same, but the traceback is printed on stderr, so there is no
    'error' output at all."""
    ip = get_ipython()
    def to_stderr(etype, evalue, stb):
        print(ip.InteractiveTB.stb2text(stb), file=sys.stderr)
    ip._showtraceback = to_stderr
    try:
        result = ip.run_cell(code, store_history=False)
    finally:
        del ip._showtraceback
    if isinstance(result.error_in_exec, KeyboardInterrupt):
        raise KeyboardInterrupt


def run_hidden(code):
    """C (fallback 2): run code, capture everything it shows (prints, errors, last expression,
    turtle drawing) and show it inside a closed HTML <details> ("Show the output"), so the cell
    can stay visible without giving the answer away."""
    import html, re
    from IPython.utils.capture import capture_output
    from IPython.display import display, HTML
    ip = get_ipython()
    def to_stderr(etype, evalue, stb):
        print(ip.InteractiveTB.stb2text(stb), file=sys.stderr)
    ip._showtraceback = to_stderr
    try:
        with capture_output() as cap:
            result = ip.run_cell(code, store_history=False)
    finally:
        del ip._showtraceback
    ansi = re.compile(r'\x1b\[[0-9;]*m')
    parts = []
    if cap.stdout:
        parts.append('<pre>' + html.escape(cap.stdout) + '</pre>')
    rich = {}
    for k, o in enumerate(cap.outputs):          # keep the last version of each display (turtle updates)
        key = (o.transient or {}).get('display_id', k) if hasattr(o, 'transient') else k
        rich[key] = o.data
    for data in rich.values():
        if 'text/html' in data:
            parts.append(data['text/html'])
        elif 'text/plain' in data:
            parts.append('<pre>' + html.escape(data['text/plain']) + '</pre>')
    if cap.stderr:
        parts.append('<pre style="color:#c00">' + html.escape(ansi.sub('', cap.stderr)) + '</pre>')
    display(HTML('<details><summary><b>Show the output</b></summary>' + ''.join(parts) + '</details>'))
    if isinstance(result.error_in_exec, KeyboardInterrupt):
        raise KeyboardInterrupt


## Part 1: which collapse metadata does Colab honour?

Test 1 is saved collapsed in all three ways (Colab list + cell metadata.id + JupyterLab key). Test 2 only through the Colab list with the top-level cell id (no metadata.id). Test 3 only with the JupyterLab key.

### Test 1: collapsed: full metadata

```python
print('LIVE 1')
```

#### Answer

SECRET 1: the output is

```text
LIVE 1
```

In [ ]:
print('LIVE 1')

#### Marker 1

In [ ]:
print('MARKER 1: Run all got past Test 1')

### Test 2: collapsed: Colab list + top-level id only

```python
print('LIVE 2')
```

#### Answer

SECRET 2: the output is

```text
LIVE 2
```

In [ ]:
print('LIVE 2')

#### Marker 2

In [ ]:
print('MARKER 2: Run all got past Test 2')

### Test 3: collapsed: JupyterLab key only

```python
print('LIVE 3')
```

#### Answer

SECRET 3: the output is

```text
LIVE 3
```

In [ ]:
print('LIVE 3')

#### Marker 3

In [ ]:
print('MARKER 3: Run all got past Test 3')

## Part 2: Run all past errors

Tests 4 and 5 (variant B: traceback printed on stderr) come first, then tests 6 and 7 (variant A: normal red error output). If Run all stops, the last MARKER with output tells which variant Colab tolerates.

### Test 4: B: syntax error, traceback on stderr

```python
x = 5
if x = 5:
    print('five')
```

#### Answer

```text
SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='?
```

In [ ]:
run_code_stderr("""
x = 5
if x = 5:
    print('five')
""")

#### Marker 4

In [ ]:
print('MARKER 4: Run all got past Test 4')

### Test 5: B: runtime error after a print, traceback on stderr

```python
print('start')
print(total)
```

#### Answer

```text
start
```

```text
NameError: name 'total' is not defined
```

In [ ]:
run_code_stderr("""
print('start')
print(total)
""")

#### Marker 5

In [ ]:
print('MARKER 5: Run all got past Test 5')

### Test 6: A: syntax error, normal error output

```python
x = 5
if x > 0
    print('positive')
```

#### Answer

```text
SyntaxError: expected ':'
```

In [ ]:
run_code("""
x = 5
if x > 0
    print('positive')
""")

#### Marker 6

In [ ]:
print('MARKER 6: Run all got past Test 6')

### Test 7: A: runtime error inside a function (traceback order), normal error output

```python
def outer(s):
    return inner(s)

def inner(s):
    return s + 1

outer('hi')
```

#### Answer

```text
TypeError: can only concatenate str (not "int") to str
```

In [ ]:
run_code("""
def outer(s):
    return inner(s)

def inner(s):
    return s + 1

outer('hi')
""")

#### Marker 7

In [ ]:
print('MARKER 7: Run all got past Test 7')

### Test 8: A: an import that fails (Colab adds a note)

```python
print('start')
import no_such_module
```

#### Answer

```text
start
```

```text
ModuleNotFoundError: No module named 'no_such_module'
```

In [ ]:
run_code("""
print('start')
import no_such_module
""")

#### Marker 8

In [ ]:
print('MARKER 8: Run all got past Test 8')

## Part 3: outputs that are not errors

### Test 9: last expression of a cell (plain cell)

```python
price = 4
price * 3
```

#### Answer

```text
12
```

In [ ]:
price = 4
price * 3

#### Marker 9

In [ ]:
print('MARKER 9: Run all got past Test 9')

### Test 10: turtle drawing (plain cell)

```python
make_turtle()
for i in range(4):
    forward(50)
    left(90)
```

#### Answer

A square.

In [ ]:
make_turtle()
for i in range(4):
    forward(50)
    left(90)

#### Marker 10

In [ ]:
print('MARKER 10: Run all got past Test 10')

## Part 4: fallback C, output inside a closed box

These cells are **not** in a collapsed section. Their output should be a closed "Show the output" box.

### Test 11: fallback C with an error and a drawing

In [ ]:
run_hidden("""
print('start')
make_turtle()
forward(80)
print(10 / 0)
""")

In [ ]:
run_hidden("""
price = 4
price * 3
""")

#### Marker 11

In [ ]:
print('MARKER 11: Run all got past Test 11')

## End

This section is not collapsed. If Run all reached the end, the next cell shows `END`.

In [ ]:
print('END: Run all reached the last cell')